In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2],
        [1, 0], [1, 1], [1, 2],
        [2, 0], [2, 1], [2, 2]]
n_edge = [(0, 1), (1, 2), 
          (3, 4), (4, 5),
          (6, 7), (7, 8),
          (0, 3), (3, 6),
          (1, 4), (4, 7),
          (2, 5), (5, 8)]
triArea = 0.5

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 9

In [ ]:
# fuseMarkers[4] = 1

In [ ]:
fuseMarkers

In [ ]:
np.array(fuseMarkers) == 1

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 0, epsilon = 1e-5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
ipu.sheet.pressure = 10

In [ ]:
ipu.numVars()

In [ ]:
vars = ipu.getVars()

In [ ]:
vars[0] = 1
vars[3] = 1
vars[4] = np.pi / 3
vars[5] = 0

In [ ]:
ipu.setVars(vars)

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
perturbation = np.random.random(ipu.numVars()) * 1e-2

In [ ]:
fd_validation.gradConvergencePlot(ipu.sheet)

In [ ]:
ipu.bent_sheet_gradient()

In [ ]:
(periodic_unit_helper.getNumpyArrayFromCSC(ipu.getPeriodicPatchToInflatableSheetMapTranspose()) @ (ipu.bent_sheet_gradient()[:-2]))

In [ ]:
ipu.gradient()

In [ ]:
fd_validation.gradConvergencePlot(ipu, perturb = perturbation)

In [ ]:
fd_validation.gradConvergencePlot(ipu, perturb = perturbation)

In [ ]:
class fd_wrapper_second:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.setVars(v)
    def numVars(self):
        return self.ipu.numVars()

    def getVars(self):
        return self.ipu.getVars()

    def energy(self):   return self.ipu.energy()
    def gradient(self): 
        gradient = np.zeros(self.numVars())
        flat_gradient = (periodic_unit_helper.getNumpyArrayFromCSC(ipu.getPeriodicPatchToInflatableSheetMapTranspose()) @ (ipu.bent_sheet_gradient()[:-2]))
        gradient[:3] = flat_gradient[:3]
        gradient[3:5] = ipu.bent_sheet_gradient()[-2:]
        gradient[5:] = flat_gradient[3:]

            
        return gradient

In [ ]:
class fd_wrapper:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.bent_sheet_setVars(v)
    def numVars(self):
        return self.ipu.bent_sheet_numVars()

    def getVars(self):
        return self.ipu.bent_sheet_getVars()

    def energy(self):   return self.ipu.energy()
    def gradient(self): return self.ipu.bent_sheet_gradient()

In [ ]:
fd_wrapper(ipu).getVars()

In [ ]:
fd_validation.gradConvergencePlot(fd_wrapper(ipu))

In [ ]:
fd_validation.gradConvergencePlot(fd_wrapper_second(ipu))